# Diagnóstico de Neumonía con CNN (versión optimizada)

Notebook autocontenido para entrenar una red neuronal convolucional sobre radiografías de tórax clasificadas como `NORMAL` o `PNEUMONIA`.

Esta versión incorpora varias mejoras sobre la arquitectura base:

- **BatchNorm + Dropout** en la red para mejor convergencia y regularización.
- **Class weights** en la función de pérdida para manejar el desbalance entre clases.
- **Scheduler de learning rate** (`ReduceLROnPlateau`) que baja el LR cuando la métrica se estanca.
- **Early stopping** que detiene el entrenamiento si no hay mejora en varias épocas.
- **Selección por recall de PNEUMONIA** en vez de accuracy: en contexto clínico es peor dejar pasar un enfermo que tener un falso positivo.
- **Data augmentation más fuerte** (affine + random erasing) para que el modelo generalice mejor.
- **Métricas finales ampliadas**: matriz de confusión y AUC-ROC.

> Uso educativo. No reemplaza evaluación médica ni validación clínica.

## 1. Importaciones y configuración

Aquí se concentran los hiperparámetros y rutas. Ajusta esta celda antes de entrenar.

**Notas:**
- En Windows/Jupyter se usa `NUM_WORKERS = 0` para evitar problemas de multiproceso.
- `PATIENCE` controla cuántas épocas sin mejora se toleran antes de detener.
- `MONITOR_METRIC` define qué se optimiza al guardar el mejor checkpoint: `"recall_pneumonia"` (recomendado para uso clínico) o `"val_acc"`.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    roc_auc_score,
)
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

# --- Reproducibilidad ---
SEED = 42

# --- Rutas ---
DATA_DIR = Path("data")
OUTPUT_DIR = Path("models")

# --- Hiperparámetros ---
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20                  # Más épocas, pero con early stopping
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4          # Regularización L2 en Adam
DROPOUT_RATE = 0.4
NUM_WORKERS = 0

# --- Estrategia de entrenamiento ---
PATIENCE = 5                 # Épocas sin mejora antes de parar
MONITOR_METRIC = "recall_pneumonia"   # o "val_acc"
USE_WEIGHTED_SAMPLER = True  # Balancea el batch en cada iteración

# --- Etiquetas ---
CLASS_TO_INDEX = {"NORMAL": 0, "PNEUMONIA": 1}
INDEX_TO_CLASS = {index: class_name for class_name, index in CLASS_TO_INDEX.items()}
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

# --- Setup ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dispositivo: {DEVICE}")
print(f"Métrica a optimizar: {MONITOR_METRIC}")

In [ ]:
# CELDA OPCIONAL PARA KAGGLE
# Ejecuta esta celda solo si estas corriendo el notebook en Kaggle.
# En Kaggle los datasets se montan en /kaggle/input y no en ./data.

KAGGLE_INPUT_ROOT = Path("/kaggle/input")

if not KAGGLE_INPUT_ROOT.exists():
    raise RuntimeError("No se detecto /kaggle/input. Omite esta celda si estas trabajando localmente.")

def has_expected_dataset_structure(path: Path) -> bool:
    required_paths = [
        path / "train" / "NORMAL",
        path / "train" / "PNEUMONIA",
        path / "val" / "NORMAL",
        path / "val" / "PNEUMONIA",
    ]
    return all(required_path.exists() for required_path in required_paths)

candidate_data_dirs = [
    path
    for path in KAGGLE_INPUT_ROOT.rglob("*")
    if path.is_dir() and has_expected_dataset_structure(path)
]

if not candidate_data_dirs:
    raise FileNotFoundError(
        "No se encontro un dataset con train/val y clases NORMAL/PNEUMONIA dentro de /kaggle/input. "
        "Revisa que el dataset este agregado como input del notebook."
    )

preferred_data_dirs = [path for path in candidate_data_dirs if path.name.lower() == "chest_xray"]
DATA_DIR = sorted(preferred_data_dirs or candidate_data_dirs, key=lambda path: str(path))[0]

print(f"DATA_DIR configurado para Kaggle: {DATA_DIR}")
print("Splits encontrados:", sorted(path.name for path in DATA_DIR.iterdir() if path.is_dir()))

## 2. Dataset y transformaciones

El dataset debe estar organizado por carpetas: `data/train/NORMAL`, `data/train/PNEUMONIA`, `data/val/NORMAL`, `data/val/PNEUMONIA` y opcionalmente `data/test/...`.

**Mejoras respecto a la versión base:**
- **Augmentation más fuerte** en entrenamiento: rotaciones más amplias, transformaciones afines y *random erasing*. Esto simula variaciones reales en radiografías (rotaciones leves del paciente, oclusiones parciales) y reduce overfitting.
- **`WeightedRandomSampler`** opcional: en cada batch se ven proporciones más balanceadas de NORMAL/PNEUMONIA. Útil porque el dataset suele tener 3x más imágenes de PNEUMONIA.

In [ ]:
@dataclass(frozen=True)
class DataLoaderConfig:
    """Configuración para construir los DataLoaders de train/val/test."""
    data_dir: Path
    batch_size: int = 32
    num_workers: int = 0
    image_size: int = 224
    pin_memory: bool = False
    use_weighted_sampler: bool = False


class PneumoniaDataset(Dataset):
    """Dataset de radiografías de tórax organizado en carpetas por clase.

    Estructura esperada:
        root_dir/
          NORMAL/
          PNEUMONIA/
    """

    def __init__(self, root_dir: str | Path, transform: transforms.Compose | None = None) -> None:
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = self._collect_samples()

        if not self.samples:
            raise ValueError(f"No se encontraron imagenes validas en: {self.root_dir}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

    def _collect_samples(self) -> list[tuple[Path, int]]:
        """Recorre cada subcarpeta de clase y arma la lista (ruta, etiqueta)."""
        samples: list[tuple[Path, int]] = []

        for class_name, class_index in CLASS_TO_INDEX.items():
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue

            for image_path in class_dir.rglob("*"):
                if image_path.suffix.lower() in VALID_EXTENSIONS:
                    samples.append((image_path, class_index))

        return sorted(samples, key=lambda sample: str(sample[0]))

    def get_labels(self) -> list[int]:
        """Devuelve solo las etiquetas, útil para calcular pesos de clase."""
        return [label for _, label in self.samples]


def build_transforms(image_size: int = 224, train: bool = True) -> transforms.Compose:
    """Construye el pipeline de transformaciones.

    En entrenamiento se aplica augmentation más agresivo para mejorar generalización.
    En validación/test solo se redimensiona y normaliza.
    """
    pipeline: list = [transforms.Resize((image_size, image_size))]

    if train:
        pipeline.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomAffine(
                degrees=15,
                translate=(0.05, 0.05),
                scale=(0.95, 1.05),
            ),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
        ])

    pipeline.extend([
        transforms.ToTensor(),
        # Normalización estándar de ImageNet (útil incluso entrenando desde cero
        # porque centra los valores y estabiliza el entrenamiento).
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    if train:
        # Random erasing: oculta parches aleatorios, simula oclusiones y
        # fuerza al modelo a no depender de regiones específicas.
        pipeline.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)))

    return transforms.Compose(pipeline)


def compute_class_weights(labels: list[int], num_classes: int = 2) -> torch.Tensor:
    """Calcula pesos inversamente proporcionales a la frecuencia de cada clase.

    Se usan en CrossEntropyLoss para penalizar más los errores en la clase minoritaria.
    """
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    weights = counts.sum() / (num_classes * counts + 1e-8)
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(labels: list[int]) -> WeightedRandomSampler:
    """Construye un sampler que muestrea cada clase con probabilidad balanceada.

    Cada muestra recibe un peso = 1 / frecuencia_de_su_clase, así los batches
    quedan aproximadamente equilibrados aunque el dataset esté desbalanceado.
    """
    counts = np.bincount(labels)
    class_weights = 1.0 / counts
    sample_weights = [class_weights[label] for label in labels]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def create_dataloader(
    split_dir: str | Path,
    batch_size: int = 32,
    image_size: int = 224,
    shuffle: bool = False,
    num_workers: int = 0,
    pin_memory: bool = False,
    train: bool = False,
    use_weighted_sampler: bool = False,
) -> DataLoader:
    """Crea un DataLoader para un split (train/val/test) específico."""
    dataset = PneumoniaDataset(
        root_dir=split_dir,
        transform=build_transforms(image_size=image_size, train=train),
    )

    sampler = None
    if train and use_weighted_sampler:
        sampler = build_weighted_sampler(dataset.get_labels())
        shuffle = False  # No se puede usar shuffle junto con sampler

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )


def create_dataloaders(config: DataLoaderConfig) -> dict[str, DataLoader]:
    """Crea los DataLoaders para los splits encontrados (train, val y opcionalmente test)."""
    split_settings = {
        "train": {"shuffle": True, "train": True, "use_weighted_sampler": config.use_weighted_sampler},
        "val": {"shuffle": False, "train": False, "use_weighted_sampler": False},
        "test": {"shuffle": False, "train": False, "use_weighted_sampler": False},
    }
    dataloaders: dict[str, DataLoader] = {}

    for split_name, settings in split_settings.items():
        split_dir = config.data_dir / split_name
        if split_dir.exists():
            dataloaders[split_name] = create_dataloader(
                split_dir=split_dir,
                batch_size=config.batch_size,
                image_size=config.image_size,
                shuffle=settings["shuffle"],
                num_workers=config.num_workers,
                pin_memory=config.pin_memory,
                train=settings["train"],
                use_weighted_sampler=settings["use_weighted_sampler"],
            )

    missing_required = {"train", "val"} - set(dataloaders)
    if missing_required:
        missing = ", ".join(sorted(missing_required))
        raise FileNotFoundError(f"Faltan splits requeridos en {config.data_dir}: {missing}")

    return dataloaders

## 3. Arquitectura mejorada: `OptimizedCNN`

Se mantiene la idea de **tres bloques convolucionales** progresivos (32 → 64 → 128 filtros), pero se le añaden mejoras clave:

| Componente | Por qué se agrega |
|---|---|
| `BatchNorm2d` después de cada Conv | Estabiliza y acelera el entrenamiento, permite usar LR más alto |
| Doble Conv por bloque | Aumenta la profundidad efectiva sin disparar el número de parámetros |
| `Dropout` en la cabeza clasificadora | Regularización contra overfitting |
| Inicialización Kaiming en Conv | Mejor punto de partida para activaciones ReLU |

El número total de parámetros sigue siendo modesto (~500K), así que la red entrena rápido incluso en CPU.

In [ ]:
class OptimizedCNN(nn.Module):
    """CNN para clasificación binaria de radiografías de tórax.

    Arquitectura:
        - 3 bloques Conv-BN-ReLU-Conv-BN-ReLU-MaxPool con 32, 64 y 128 filtros.
        - AdaptiveAvgPool a 1x1 (permite cualquier tamaño de entrada).
        - Clasificador con Dropout y dos capas Linear.

    Args:
        num_classes: Número de clases de salida.
        dropout_rate: Probabilidad de Dropout en la cabeza clasificadora.
    """

    def __init__(self, num_classes: int = 2, dropout_rate: float = 0.4) -> None:
        super().__init__()

        self.features = nn.Sequential(
            self._conv_block(3, 32),
            self._conv_block(32, 64),
            self._conv_block(64, 128),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )

        self._initialize_weights()

    @staticmethod
    def _conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
        """Bloque Conv-BN-ReLU-Conv-BN-ReLU-MaxPool.

        Dos convoluciones consecutivas aumentan la capacidad sin reducir resolución,
        y el MaxPool al final divide a la mitad el tamaño espacial.
        """
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

    def _initialize_weights(self) -> None:
        """Inicialización Kaiming para Conv y Xavier para Linear."""
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.constant_(module.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        return self.classifier(x)


model = OptimizedCNN(num_classes=len(CLASS_TO_INDEX), dropout_rate=DROPOUT_RATE).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
model

## 4. Funciones de entrenamiento y evaluación

**Mejoras:**
- `evaluate` ahora también devuelve las **probabilidades** (para calcular AUC-ROC) y el **recall específico de PNEUMONIA**.
- `plot_history` muestra además la evolución del recall de la clase positiva.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """Ejecuta una época completa de entrenamiento.

    Returns:
        (loss promedio, accuracy) sobre todo el split de train.
    """
    model.train()
    running_loss = 0.0
    all_predictions: list[int] = []
    all_targets: list[int] = []

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = torch.argmax(outputs, dim=1)
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().tolist())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_accuracy = accuracy_score(all_targets, all_predictions)
    return epoch_loss, epoch_accuracy


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    """Evalúa el modelo y devuelve un dict con todas las métricas relevantes.

    Returns:
        dict con keys: loss, acc, recall_pneumonia, predictions, targets, probabilities.
    """
    model.eval()
    running_loss = 0.0
    all_predictions: list[int] = []
    all_targets: list[int] = []
    all_probabilities: list[float] = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            # Probabilidad de la clase PNEUMONIA (índice 1)
            probabilities = torch.softmax(outputs, dim=1)[:, 1]
            predictions = torch.argmax(outputs, dim=1)

            all_predictions.extend(predictions.cpu().tolist())
            all_targets.extend(labels.cpu().tolist())
            all_probabilities.extend(probabilities.cpu().tolist())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_accuracy = accuracy_score(all_targets, all_predictions)
    # Recall de la clase positiva (PNEUMONIA): qué porcentaje de enfermos detectamos
    recall_pneumonia = recall_score(all_targets, all_predictions, pos_label=1, zero_division=0)

    return {
        "loss": epoch_loss,
        "acc": epoch_accuracy,
        "recall_pneumonia": recall_pneumonia,
        "predictions": all_predictions,
        "targets": all_targets,
        "probabilities": all_probabilities,
    }


def plot_history(history: dict[str, list[float]], output_path: Path | None = None) -> None:
    """Grafica las curvas de pérdida, accuracy y recall de pneumonia."""
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"], label="Validación")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Pérdida")
    axes[0].set_title("Pérdida")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], label="Train")
    axes[1].plot(epochs, history["val_acc"], label="Validación")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("Exactitud")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, history["val_recall_pneumonia"], color="darkred", label="Val")
    axes[2].set_xlabel("Época")
    axes[2].set_ylabel("Recall")
    axes[2].set_title("Recall PNEUMONIA (validación)")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    if output_path is not None:
        plt.savefig(output_path, dpi=120)
    plt.show()


def plot_confusion_matrix(targets: list[int], predictions: list[int], output_path: Path | None = None) -> None:
    """Visualiza la matriz de confusión sobre el set de test."""
    cm = confusion_matrix(targets, predictions)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["NORMAL", "PNEUMONIA"])
    ax.set_yticklabels(["NORMAL", "PNEUMONIA"])
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title("Matriz de confusión")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    if output_path is not None:
        plt.savefig(output_path, dpi=120)
    plt.show()

## 5. Carga de datos

Esta celda fallará con un mensaje claro si aún no existe la carpeta `data/` con los splits requeridos.

In [ ]:
dataloaders = create_dataloaders(
    DataLoaderConfig(
        data_dir=DATA_DIR,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        image_size=IMAGE_SIZE,
        pin_memory=DEVICE.type == "cuda",
        use_weighted_sampler=USE_WEIGHTED_SAMPLER,
    )
)

# Inspección rápida del balance de clases en train
train_labels = dataloaders["train"].dataset.get_labels()
class_counts = np.bincount(train_labels, minlength=2)

print("Distribución por split:")
for split_name, dataloader in dataloaders.items():
    print(f"  {split_name}: {len(dataloader.dataset)} imágenes")

print(f"\nBalance en train: NORMAL={class_counts[0]} | PNEUMONIA={class_counts[1]} "
      f"(ratio {class_counts[1]/max(class_counts[0],1):.2f}x)")

## 6. Entrenamiento con scheduler y early stopping

**Estrategia:**
1. **CrossEntropyLoss con `class_weights`**: penaliza más fuerte los errores en la clase minoritaria.
2. **Adam con `weight_decay`**: agrega regularización L2.
3. **`ReduceLROnPlateau`**: baja el learning rate ×0.5 cuando la métrica de validación se estanca 2 épocas.
4. **Early stopping**: si la métrica monitoreada no mejora en `PATIENCE` épocas, se detiene.
5. **Mejor checkpoint** = el que maximiza la métrica monitoreada (por defecto recall de PNEUMONIA).

In [ ]:
# Pesos de clase calculados sobre el set de entrenamiento
train_labels = dataloaders["train"].dataset.get_labels()
class_weights = compute_class_weights(train_labels).to(DEVICE)
print(f"Class weights: NORMAL={class_weights[0]:.3f} | PNEUMONIA={class_weights[1]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",            # Monitoreamos una métrica que queremos maximizar
    factor=0.5,
    patience=2,
)

best_metric = -1.0
epochs_without_improvement = 0
best_model_path = OUTPUT_DIR / "best_optimized_cnn.pth"
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_recall_pneumonia": [],
}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=dataloaders["train"],
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
    )
    val_results = evaluate(
        model=model,
        dataloader=dataloaders["val"],
        criterion=criterion,
        device=DEVICE,
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_results["loss"])
    history["val_acc"].append(val_results["acc"])
    history["val_recall_pneumonia"].append(val_results["recall_pneumonia"])

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Época {epoch:03d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_results['loss']:.4f} val_acc={val_results['acc']:.4f} "
        f"val_recall_pneu={val_results['recall_pneumonia']:.4f} | "
        f"lr={current_lr:.2e}"
    )

    # Métrica que se usa para guardar el mejor modelo y para early stopping
    monitored_value = val_results[MONITOR_METRIC] if MONITOR_METRIC in val_results else val_results["acc"]
    scheduler.step(monitored_value)

    if monitored_value > best_metric:
        best_metric = monitored_value
        epochs_without_improvement = 0
        torch.save(
            {
                "model_name": "optimized_cnn",
                "model_state_dict": model.state_dict(),
                "best_metric": best_metric,
                "monitored_metric": MONITOR_METRIC,
                "image_size": IMAGE_SIZE,
                "class_to_index": CLASS_TO_INDEX,
            },
            best_model_path,
        )
        print(f"  ✔ Nuevo mejor modelo ({MONITOR_METRIC}={best_metric:.4f}) guardado en: {best_model_path}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping: {PATIENCE} épocas sin mejora.")
            break

print(f"\nMejor {MONITOR_METRIC} en validación: {best_metric:.4f}")

## 7. Curvas de entrenamiento

Las curvas permiten detectar visualmente:
- **Overfitting**: si `train_loss` baja pero `val_loss` sube.
- **Underfitting**: si ambas se mantienen altas.
- **Convergencia**: si las curvas se aplanan, ya no vale la pena seguir entrenando.

In [ ]:
plot_history(history, OUTPUT_DIR / "training_curves.png")

## 8. Evaluación final en test

Se carga el mejor checkpoint y se reportan:
- **Classification report**: precision, recall y F1 por clase.
- **Matriz de confusión**: distribución de aciertos y errores.
- **AUC-ROC**: capacidad discriminativa del modelo independiente del umbral.

In [ ]:
if best_model_path.exists():
    checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Checkpoint cargado. Mejor {checkpoint['monitored_metric']} en validación: {checkpoint['best_metric']:.4f}\n")

if "test" in dataloaders:
    test_results = evaluate(
        model=model,
        dataloader=dataloaders["test"],
        criterion=criterion,
        device=DEVICE,
    )

    print(f"Test loss = {test_results['loss']:.4f}")
    print(f"Test acc  = {test_results['acc']:.4f}")
    print(f"Test recall PNEUMONIA = {test_results['recall_pneumonia']:.4f}")

    try:
        auc = roc_auc_score(test_results["targets"], test_results["probabilities"])
        print(f"Test AUC-ROC = {auc:.4f}")
    except ValueError:
        print("AUC-ROC no calculable (solo una clase en el split).")

    print("\n" + classification_report(
        test_results["targets"],
        test_results["predictions"],
        target_names=["NORMAL", "PNEUMONIA"],
        digits=4,
    ))

    plot_confusion_matrix(
        test_results["targets"],
        test_results["predictions"],
        OUTPUT_DIR / "confusion_matrix.png",
    )
else:
    print("No se encontró split test. Evaluación final omitida.")

## 9. Próximos pasos posibles

Para subir aún más las métricas:

- **Transfer learning** con ResNet18, EfficientNet-B0 o DenseNet121 preentrenados en ImageNet (gana fácil 3–5 puntos de F1).
- **Mixup o CutMix** como augmentation extra.
- **Test-time augmentation (TTA)**: promediar predicciones sobre varias versiones aumentadas de cada imagen de test.
- **Calibración del umbral**: en vez de usar 0.5 como corte, optimizarlo según el costo clínico (false negative >> false positive).
- **Ensembles**: promediar varios modelos entrenados con seeds distintas.